In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import colorsys
from adjustText import adjust_text
import cmcrameri.cm as cmc

import scanpy as sc
import anndata as ad
import squidpy as sq

In [ ]:
plt.rcdefaults()

plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "figure.dpi":         300,
})

In [ ]:
OUT_DIR = "output"
FIG_DIR = "figures"
ADATA = "../../quality_control/primary-cohort/adata.h5ad"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
if not os.path.exists(FIG_DIR):
    os.makedirs(FIG_DIR)
sc.settings.figdir = FIG_DIR
plt.rcParams['savefig.dpi'] = 600

In [ ]:
adata = ad.read_h5ad(ADATA)
adata.obs['class'] = adata.obs['class'].astype(str)
adata.obs['class_overall'] = adata.obs['class'].replace({'OvaryR': 'Ovary', 'OvaryL': 'Ovary'})
adata.obs['PFI'] = adata.obs.PFI.astype(str)
adata.obs['PFI_short_long'] = adata.obs['PFI'].replace({'short': 'short', 'medium': 'long', 'long': 'long'})
adata.obs['patient'] = adata.obs['patient'].astype(str)
adata.obs['anno'] = adata.obs['class'] + '_' + adata.obs['patient'] + '_PFI-' + adata.obs['PFI']

adata_tumor = adata[adata.obs['histology'].isin(['Tumor Epithelium'])].copy()
adata_tumor = adata_tumor[~adata_tumor.obs['class'].isin(['Marker'])].copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata_tumor,
    n_top_genes=2000,
    flavor="seurat_v3",
    subset=False,
    layer='counts',
)
sc.tl.rank_genes_groups(
    adata_tumor,
    layer='log-transformed',
    groupby="PFI",
    method="wilcoxon",
    use_raw=False,
)

In [ ]:
n_genes = 10
short_markers = adata_tumor.uns['rank_genes_groups']['names']['short'][:n_genes].tolist()
medium_markers = adata_tumor.uns['rank_genes_groups']['names']['medium'][:n_genes].tolist()
long_markers  = adata_tumor.uns['rank_genes_groups']['names']['long'][:n_genes].tolist()

medium_markers_unique = [g for g in medium_markers if g not in short_markers]
long_markers_unique = [g for g in long_markers if g not in short_markers]

var_names = {
    'short PFI': short_markers,
    'medium PFI': medium_markers_unique,
    'long PFI':  long_markers_unique,
}

adata_tumor.obs['PFI'] = pd.Categorical(
    adata_tumor.obs['PFI'],
    categories=['short', 'medium', 'long'],
    ordered=True,
)

with plt.rc_context({
    'font.size': 13,
    'axes.labelsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 13,
}):
    sc.pl.rank_genes_groups_matrixplot(
        adata_tumor,
        var_names=var_names,
        groupby="PFI",
        values_to_plot="logfoldchanges",
        cmap=cmc.lipari,
        vmin=-3, vmax=3,
        var_group_rotation=0,
        show=False,
    )

    mp_axes = plt.gcf().axes
    for ax in mp_axes:
        for label in ax.get_yticklabels():
            label.set_fontsize(13)
        for txt in ax.texts:
            txt.set_fontsize(13)
        
    genes_to_highlight = {'C3', 'IFI27', 'COL13A1'}
    for ax in plt.gcf().axes:
        for label in ax.get_xticklabels():
            if label.get_text() in genes_to_highlight:
                label.set_fontweight('bold')

    plt.savefig(os.path.join(FIG_DIR, "marker_genes_matrixplot.png"), dpi=600, bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, "marker_genes_matrixplot.pdf"), bbox_inches='tight')
    plt.show()

### Adnex

In [ ]:
adnex = adata_tumor[adata_tumor.obs['class_overall'] == 'Ovary', :].copy()

sc.pp.highly_variable_genes(
    adnex,
    n_top_genes=2000,
    flavor="seurat_v3",
    subset=False,
    layer='counts',
)
sc.tl.rank_genes_groups(
    adnex,
    layer='log-transformed',
    groupby="PFI",
    method="wilcoxon",
    use_raw=False,
)

In [ ]:
adnex.uns['rank_genes_groups']['names']['short'].tolist()[:30]

In [ ]:
n_genes = 10
short_markers = adnex.uns['rank_genes_groups']['names']['short'][:n_genes].tolist()
medium_markers = adnex.uns['rank_genes_groups']['names']['medium'][:n_genes].tolist()
long_markers  = adnex.uns['rank_genes_groups']['names']['long'][:n_genes].tolist()

medium_markers_unique = [g for g in medium_markers if g not in short_markers]
long_markers_unique = [g for g in long_markers if g not in short_markers]

var_names = {
    'short PFI': short_markers,
    'medium PFI': medium_markers_unique,
    'long PFI':  long_markers_unique,
}

with plt.rc_context({
    'font.size': 13,
    'axes.labelsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 13,
}):
    sc.pl.rank_genes_groups_matrixplot(
        adnex,
        var_names=var_names,
        groupby="PFI",
        values_to_plot="logfoldchanges",
        cmap=cmc.lipari,
        vmin=-3, vmax=3,
        var_group_rotation=0,
        show=False,
    )

    mp_axes = plt.gcf().axes
    for ax in mp_axes:
        for label in ax.get_yticklabels():
            label.set_fontsize(13)
        for txt in ax.texts:
            txt.set_fontsize(13)
            
    genes_to_highlight = {'C3', 'IFI27', 'COL13A1'}

    for ax in plt.gcf().axes:
        for label in ax.get_xticklabels():
            if label.get_text() in genes_to_highlight:
                label.set_fontweight('bold')

plt.savefig(os.path.join(FIG_DIR, "adnex_marker_genes_matrixplot.png"), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, "adnex_marker_genes_matrixplot.pdf"), bbox_inches='tight')
plt.show()

In [ ]:
omentum = adata_tumor[adata_tumor.obs['class_overall'] == 'Omentum', :].copy()

sc.pp.highly_variable_genes(
    omentum,
    n_top_genes=2000,
    flavor="seurat_v3",
    subset=False,
    layer='counts',
)
sc.tl.rank_genes_groups(
    omentum,
    layer='log-transformed',
    groupby="PFI",
    method="wilcoxon",
    use_raw=False,
)

In [ ]:
omentum.uns['rank_genes_groups']['names']['short'].tolist()[:30]

In [ ]:
n_genes = 10
short_markers = omentum.uns['rank_genes_groups']['names']['short'][:n_genes].tolist()
medium_markers = omentum.uns['rank_genes_groups']['names']['medium'][:n_genes].tolist()
long_markers  = omentum.uns['rank_genes_groups']['names']['long'][:n_genes].tolist()

medium_markers_unique = [g for g in medium_markers if g not in short_markers]
long_markers_unique = [g for g in long_markers if g not in short_markers]

var_names = {
    'short PFI': short_markers,
    'medium PFI': medium_markers_unique,
    'long PFI': long_markers_unique,
}

with plt.rc_context({
    'font.size': 13,
    'axes.labelsize': 13,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 13,
}):
    sc.pl.rank_genes_groups_matrixplot(
        omentum,
        var_names=var_names,
        groupby="PFI",
        values_to_plot="logfoldchanges",
        cmap=cmc.lipari,
        vmin=-3, vmax=3,
        var_group_rotation=0,
        show=False,
    )

    mp_axes = plt.gcf().axes
    for ax in mp_axes:
        for label in ax.get_yticklabels():
            label.set_fontsize(13)
        for txt in ax.texts:
            txt.set_fontsize(13)

    genes_to_highlight = {'C3', 'IFI27'}

    for ax in plt.gcf().axes:
        for label in ax.get_xticklabels():
            if label.get_text() in genes_to_highlight:
                label.set_fontweight('bold')

plt.savefig(os.path.join(FIG_DIR, "omentum_marker_genes_matrixplot.png"), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, "omentum_marker_genes_matrixplot.pdf"), bbox_inches='tight')
plt.show()